In [ ]:
import os
import sqlite3
import pandas as pd

ANHOS = [2023, 2024, 2025]
RUTA_PROCESADOS = os.path.join("data", "processed")
DB_PATH = os.path.join("contrataciones.db")

ELIMINAR_HUERFANOS = False     # True borra filas sin padre; False solo las reporta
PARTIES_POR_PARTY_ID = False   # True = una fila por party_id, como en cargar_sqlite.py

COLUMNAS = {
    "records": {
        "ocid": "ocid",
        "compiledRelease/date": "release_date",
        "compiledRelease/buyer/id": "buyer_id",
        "compiledRelease/buyer/name": "buyer_name",
        "compiledRelease/planning/budget/amount/amount": "budget_amount",
        "compiledRelease/tender/id": "tender_id",
        "compiledRelease/tender/title": "tender_title",
        "compiledRelease/tender/procuringEntity/id": "procuring_entity_id",
        "compiledRelease/tender/procuringEntity/name": "procuring_entity_name",
        "compiledRelease/tender/datePublished": "tender_published_date",
        "compiledRelease/tender/procurementMethod": "procurement_method",
        "compiledRelease/tender/mainProcurementCategory": "procurement_category",
        "compiledRelease/tender/value/amount": "tender_value_amount",
        "compiledRelease/tender/value/currency": "tender_value_currency",
        "compiledRelease/tender/value/amount_PEN": "tender_value_amount_pen",
        "compiledRelease/tender/numberOfTenderers": "number_of_tenderers",
        "compiledRelease/tender/tenderPeriod/startDate": "tender_start_date",
        "compiledRelease/tender/tenderPeriod/endDate": "tender_end_date",
    },
    "com_awards": {
        "ocid": "ocid",
        "compiledRelease/awards/0/id": "award_id",
        "compiledRelease/awards/0/value/amount": "award_amount",
        "compiledRelease/awards/0/value/currency": "award_currency",
        "compiledRelease/awards/0/date": "award_date",
    },
    "com_awa_suppliers": {
        "ocid": "ocid",
        "compiledRelease/awards/0/id": "award_id",
        "compiledRelease/awards/0/suppliers/0/id": "supplier_id",
        "compiledRelease/awards/0/suppliers/0/name": "supplier_name",
    },
    "com_contracts": {
        "ocid": "ocid",
        "compiledRelease/contracts/0/id": "contract_id",
        "compiledRelease/contracts/0/awardID": "award_id",
        "compiledRelease/contracts/0/title": "contract_title",
        "compiledRelease/contracts/0/value/amount": "contract_amount",
        "compiledRelease/contracts/0/dateSigned": "date_signed",
        "compiledRelease/contracts/0/period/startDate": "period_start_date",
        "compiledRelease/contracts/0/period/endDate": "period_end_date",
        "compiledRelease/contracts/0/implementation/finalValue/amount": "final_value_amount",
    },
    "com_parties": {
        "ocid": "ocid",
        "compiledRelease/parties/0/id": "party_id",
        "compiledRelease/parties/0/name": "party_name",
        "compiledRelease/parties/0/identifier/id": "identifier_id",
        "compiledRelease/parties/0/identifier/scheme": "identifier_scheme",
        "compiledRelease/parties/0/roles": "roles",
        "compiledRelease/parties/0/address/region": "region",
        "compiledRelease/parties/0/address/department": "department",
    },
    "com_ten_tenderers": {
        "ocid": "ocid",
        "compiledRelease/tender/id": "tender_id",
        "compiledRelease/tender/tenderers/0/id": "tenderer_id",
        "compiledRelease/tender/tenderers/0/name": "tenderer_name",
    },
}

## Funciones de apoyo

In [2]:
NULOS = {"", "nan", "none", "null", "n/a", "na", "-", "s/n"}


def texto(s, mayus=False):
    # espacios repetidos fuera, vacíos y marcadores tipo "NaN" pasan a nulo
    s = s.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.mask(s.str.lower().isin(NULOS))
    return s.str.upper() if mayus else s


def codigo(s):
    # ids como texto, quita el ".0" que deja un paso previo por float
    return texto(s).str.replace(r"\.0$", "", regex=True)


def monto(s, nombre=None):
    # numérico, un monto negativo no es válido y se anula
    n = pd.to_numeric(s, errors="coerce")
    negativos = (n < 0).sum()
    if nombre and negativos:
        print(f"  {nombre}: {negativos:,} valores negativos anulados")
    return n.mask(n < 0)


def entero(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")


def fecha(s):
    # con zona horaria -> hora de Lima. Sin zona se asume que ya viene en hora local
    s = s.astype("string").str.strip()
    con_zona = s.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", na=False)
    f_zona = pd.to_datetime(s.where(con_zona), errors="coerce", utc=True, format="mixed")
    f_zona = f_zona.dt.tz_convert("America/Lima").dt.tz_localize(None)
    f_local = pd.to_datetime(s.where(~con_zona), errors="coerce", format="mixed")
    return f_zona.fillna(f_local)


def llave(*cols):
    out = cols[0]
    for c in cols[1:]:
        out = out + "-" + c
    return out


def cargar(tipo):
    # une los archivos de todos los años y marca el año de origen
    mapa = COLUMNAS[tipo]
    partes = []
    for anho in ANHOS:
        ruta = os.path.join(RUTA_PROCESADOS, f"{tipo}_{anho}_completo.csv")
        df = pd.read_csv(ruta, usecols=list(mapa), dtype=str).rename(columns=mapa)
        df["anho"] = anho
        partes.append(df)
    out = pd.concat(partes, ignore_index=True)
    out["anho"] = out["anho"].astype("Int16")
    print(f"{tipo}: {len(out):,} filas leídas")
    return out


def depurar(df, pk, nombre, orden=("anho",)):
    # sin key-> fuera. key repetida (p. ej. entre años) -> queda la versión más reciente
    n0 = len(df)
    df = df[df[pk].notna()]
    n1 = len(df)
    df = df.sort_values(list(orden), na_position="first").drop_duplicates(pk, keep="last")
    print(f"{nombre}: {n0 - n1:,} sin llave, {n1 - len(df):,} repetidas -> {len(df):,} filas")
    return df.reset_index(drop=True)

## records

In [3]:
rec = cargar("records")

rec["ocid"] = texto(rec["ocid"])
for c in ["buyer_id", "tender_id", "procuring_entity_id"]:
    rec[c] = codigo(rec[c])

for c in ["buyer_name", "procuring_entity_name"]:
    rec[c] = texto(rec[c], mayus=True)
rec["tender_title"] = texto(rec["tender_title"])
rec["tender_value_currency"] = texto(rec["tender_value_currency"], mayus=True)
for c in ["procurement_method", "procurement_category"]:
    rec[c] = texto(rec[c]).str.lower()

for c in ["budget_amount", "tender_value_amount"]:
    rec[c] = monto(rec[c], c)
    
rec["tender_value_amount_pen"] = pd.to_numeric(rec["tender_value_amount_pen"], errors="coerce")
rec["number_of_tenderers"] = entero(rec["number_of_tenderers"])

for c in ["release_date", "tender_published_date", "tender_start_date", "tender_end_date"]:
    rec[c] = fecha(rec[c])

# derivadas
rec["tender_duration_days"] = (rec["tender_end_date"] - rec["tender_start_date"]).dt.days.astype("Int64")
rec["tender_year"] = rec["tender_published_date"].dt.year.astype("Int16")
rec["periodo_invalido"] = rec["tender_duration_days"].lt(0).fillna(False)

records = depurar(rec, "ocid", "records", orden=("anho", "release_date"))
records.dtypes

records: 236,234 filas leídas
records: 0 sin llave, 0 repetidas -> 236,234 filas


ocid                       string[python]
release_date               datetime64[ns]
buyer_id                   string[python]
buyer_name                 string[python]
budget_amount                     float64
tender_id                  string[python]
tender_title               string[python]
procuring_entity_id        string[python]
procuring_entity_name      string[python]
tender_published_date      datetime64[ns]
procurement_method         string[python]
procurement_category       string[python]
tender_value_amount               float64
tender_value_currency      string[python]
tender_value_amount_pen           float64
tender_start_date          datetime64[ns]
tender_end_date            datetime64[ns]
number_of_tenderers                 Int64
anho                                Int16
tender_duration_days                Int64
tender_year                         Int16
periodo_invalido                  boolean
dtype: object

## com_awards

In [4]:
aw = cargar("com_awards")

aw["ocid"] = texto(aw["ocid"])
aw["award_id"] = codigo(aw["award_id"])
aw["award_key"] = llave(aw["ocid"], aw["award_id"])

aw["award_amount"] = monto(aw["award_amount"], "award_amount")
aw["award_currency"] = texto(aw["award_currency"], mayus=True)
aw["award_date"] = fecha(aw["award_date"])

com_awards = depurar(aw, "award_key", "com_awards")
com_awards = com_awards[
    ["award_key", "ocid", "award_id", "award_amount", "award_currency", "award_date", "anho"]
]
com_awards.dtypes

com_awards: 203,498 filas leídas
com_awards: 0 sin llave, 0 repetidas -> 203,498 filas


award_key         string[python]
ocid              string[python]
award_id          string[python]
award_amount             float64
award_currency    string[python]
award_date        datetime64[ns]
anho                       Int16
dtype: object

## com_awa_suppliers

In [5]:
sup = cargar("com_awa_suppliers")

sup["ocid"] = texto(sup["ocid"])
for c in ["award_id", "supplier_id"]:
    sup[c] = codigo(sup[c])
sup["supplier_name"] = texto(sup["supplier_name"], mayus=True)

sup["award_key"] = llave(sup["ocid"], sup["award_id"])
sup["supplier_key"] = llave(sup["ocid"], sup["award_id"], sup["supplier_id"])

com_awa_suppliers = depurar(sup, "supplier_key", "com_awa_suppliers")
com_awa_suppliers = com_awa_suppliers[
    ["supplier_key", "award_key", "ocid", "award_id", "supplier_id", "supplier_name", "anho"]
]
com_awa_suppliers.dtypes

com_awa_suppliers: 199,581 filas leídas
com_awa_suppliers: 0 sin llave, 0 repetidas -> 199,581 filas


supplier_key     string[python]
award_key        string[python]
ocid             string[python]
award_id         string[python]
supplier_id      string[python]
supplier_name    string[python]
anho                      Int16
dtype: object

## com_contracts

In [6]:
con = cargar("com_contracts")

con["ocid"] = texto(con["ocid"])
for c in ["contract_id", "award_id"]:
    con[c] = codigo(con[c])
con["contract_title"] = texto(con["contract_title"])

con["contract_key"] = llave(con["ocid"], con["contract_id"])
con["award_key"] = llave(con["ocid"], con["award_id"])

for c in ["contract_amount", "final_value_amount"]:
    con[c] = monto(con[c], c)
for c in ["date_signed", "period_start_date", "period_end_date"]:
    con[c] = fecha(con[c])

# derivadas
con["period_days"] = (con["period_end_date"] - con["period_start_date"]).dt.days.astype("Int64")
con["periodo_invalido"] = con["period_days"].lt(0).fillna(False)
con["variacion_monto"] = con["final_value_amount"] - con["contract_amount"]

com_contracts = depurar(con, "contract_key", "com_contracts")
com_contracts = com_contracts[
    ["contract_key", "award_key", "ocid", "contract_id", "award_id", "contract_title",
     "contract_amount", "final_value_amount", "variacion_monto", "date_signed",
     "period_start_date", "period_end_date", "period_days", "periodo_invalido", "anho"]
]
com_contracts.dtypes

com_contracts: 196,827 filas leídas
com_contracts: 0 sin llave, 0 repetidas -> 196,827 filas


contract_key          string[python]
award_key             string[python]
ocid                  string[python]
contract_id           string[python]
award_id              string[python]
contract_title        string[python]
contract_amount              float64
final_value_amount           float64
variacion_monto              float64
date_signed           datetime64[ns]
period_start_date     datetime64[ns]
period_end_date       datetime64[ns]
period_days                    Int64
periodo_invalido             boolean
anho                           Int16
dtype: object

## com_parties

In [7]:
par = cargar("com_parties")

par["ocid"] = texto(par["ocid"])
for c in ["party_id", "identifier_id"]:
    par[c] = codigo(par[c])
par["party_name"] = texto(par["party_name"], mayus=True)
for c in ["identifier_scheme", "region", "department"]:
    par[c] = texto(par[c], mayus=True)

par["party_key"] = llave(par["ocid"], par["party_id"])

# roles llegan como lista en texto; queda una cadena simple y una bandera por rol
roles = (
    texto(par["roles"])
    .str.replace(r"""[\[\]'"]""", "", regex=True)
    .str.replace(r"\s*,\s*", ",", regex=True)
    .str.lower()
)
par["roles"] = roles
for r in ["buyer", "procuringentity", "supplier", "tenderer"]:
    par["es_" + r] = roles.str.contains(r, na=False)

# modo catálogo: un registro por party_id (el más reciente); el ocid solo apunta a un proceso
if PARTIES_POR_PARTY_ID:
    par = par[par["party_id"].notna()].sort_values("anho").drop_duplicates("party_id", keep="last")

com_parties = depurar(par, "party_key", "com_parties")
com_parties = com_parties[
    ["party_key", "ocid", "party_id", "party_name", "identifier_id", "identifier_scheme",
     "roles", "es_buyer", "es_procuringentity", "es_supplier", "es_tenderer",
     "region", "department", "anho"]
]
com_parties.dtypes

com_parties: 3,143,082 filas leídas
com_parties: 0 sin llave, 0 repetidas -> 3,143,082 filas


party_key             string[python]
ocid                  string[python]
party_id              string[python]
party_name            string[python]
identifier_id         string[python]
identifier_scheme     string[python]
roles                 string[python]
es_buyer                     boolean
es_procuringentity           boolean
es_supplier                  boolean
es_tenderer                  boolean
region                string[python]
department            string[python]
anho                           Int16
dtype: object

## com_ten_tenderers

In [8]:
ten = cargar("com_ten_tenderers")

ten["ocid"] = texto(ten["ocid"])
for c in ["tender_id", "tenderer_id"]:
    ten[c] = codigo(ten[c])
ten["tenderer_name"] = texto(ten["tenderer_name"], mayus=True)

ten["tenderer_key"] = llave(ten["ocid"], ten["tenderer_id"])


com_ten_tenderers = depurar(ten, "tenderer_key", "com_ten_tenderers")
com_ten_tenderers = com_ten_tenderers[
    ["tenderer_key", "ocid", "tender_id", "tenderer_id", "tenderer_name", "anho"]
]
com_ten_tenderers.dtypes

com_ten_tenderers: 2,906,843 filas leídas
com_ten_tenderers: 0 sin llave, 0 repetidas -> 2,906,843 filas


tenderer_key     string[python]
ocid             string[python]
tender_id        string[python]
tenderer_id      string[python]
tenderer_name    string[python]
anho                      Int16
dtype: object

## Integridad referencial

In [9]:
def huerfanos(hijo, fk, padre, pk, nombre):
    sin_padre = ~hijo[fk].isin(padre[pk])
    print(f"{nombre}: {sin_padre.sum():,} filas sin padre en {fk}")
    return hijo[~sin_padre] if ELIMINAR_HUERFANOS else hijo


com_parties = huerfanos(com_parties, "ocid", records, "ocid", "com_parties")
com_ten_tenderers = huerfanos(com_ten_tenderers, "ocid", records, "ocid", "com_ten_tenderers")
com_awards = huerfanos(com_awards, "ocid", records, "ocid", "com_awards")
# awards va antes: contratos y proveedores se validan contra lo que ya quedó
com_contracts = huerfanos(com_contracts, "award_key", com_awards, "award_key", "com_contracts")
com_awa_suppliers = huerfanos(com_awa_suppliers, "award_key", com_awards, "award_key", "com_awa_suppliers")

com_parties: 0 filas sin padre en ocid
com_ten_tenderers: 0 filas sin padre en ocid
com_awards: 0 filas sin padre en ocid
com_contracts: 1,933 filas sin padre en award_key
com_awa_suppliers: 0 filas sin padre en award_key


In [10]:
# Investigación: contratos sin adjudicación en com_awards
print("Revisado:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))

sin_award = com_contracts[~com_contracts["award_key"].isin(com_awards["award_key"])]
n_proc = sin_award["ocid"].nunique()


# 1) tamaño del problema
resumen = pd.Series({
    "contratos_sin_award": len(sin_award),
    "pct_contratos": round(100 * len(sin_award) / len(com_contracts), 2),
    "procesos_afectados": n_proc,
    "pct_procesos_con_contratos": round(100 * n_proc / com_contracts["ocid"].nunique(), 2),
    "pct_monto": round(100 * sin_award["contract_amount"].sum() / com_contracts["contract_amount"].sum(), 2),
})
print(resumen.to_string())

# 2) concentración por proceso y adjudicaciones que sí se cargaron
top = (
    sin_award.groupby("ocid").size().rename("contratos_sin_award").to_frame()
    .join(com_awards.groupby("ocid").size().rename("awards_cargados"))
    .fillna({"awards_cargados": 0})
    .sort_values("contratos_sin_award", ascending=False)
)
top["awards_cargados"] = top["awards_cargados"].astype(int)
top["pct_acumulado"] = (top["contratos_sin_award"].cumsum() / top["contratos_sin_award"].sum() * 100).round(1)
display(top.head(20))

# 3) ejemplos: award_id que pide el contrato vs los que sí están en awards
cargados = com_awards.groupby("ocid")["award_id"].agg(lambda s: ", ".join(s))
ejemplos = sin_award[["ocid", "award_id"]].head(10).copy()
ejemplos["awards_cargados"] = ejemplos["ocid"].map(cargados)
display(ejemplos)


# contratos cuya adjudicación no está en com_awards (se conservan; ver nota de calidad)
com_contracts = com_contracts.assign(
    award_en_awards=com_contracts["award_key"].isin(com_awards["award_key"])
)

Revisado: 2026-10-03 05:04
contratos_sin_award           1933.00
pct_contratos                    0.98
procesos_afectados             663.00
pct_procesos_con_contratos       0.39
pct_monto                        0.30


,contratos_sin_award,awards_cargados,pct_acumulado
ocid,,,
ocds-dgv273-seacev3-1072867,289,8,15.0
ocds-dgv273-seacev3-1076786,199,9,25.2
ocds-dgv273-seacev3-1072061,147,10,32.9
ocds-dgv273-seacev3-1075223,106,6,38.3
ocds-dgv273-seacev3-2024-2543-1416,48,1,40.8
ocds-dgv273-seacev3-2023-200404-5,43,1,43.0
ocds-dgv273-seacev3-1168067,29,1,44.5
ocds-dgv273-seacev3-1122087,26,1,45.9
ocds-dgv273-seacev3-2024-857-12,18,3,46.8


,ocid,award_id,awards_cargados
223,ocds-dgv273-seacev3-937629,937629-20609735601,937629-20530147551
641,ocds-dgv273-seacev3-2023-1068-31,936246-591232,"936246-591233, 936246-591234, 936246-591242, 9..."
642,ocds-dgv273-seacev3-2023-1068-31,936246-591232,"936246-591233, 936246-591234, 936246-591242, 9..."
643,ocds-dgv273-seacev3-2023-1068-31,936246-591239,"936246-591233, 936246-591234, 936246-591242, 9..."
644,ocds-dgv273-seacev3-2023-1068-31,936246-591239,"936246-591233, 936246-591234, 936246-591242, 9..."
646,ocds-dgv273-seacev3-2023-1068-31,936246-591240,"936246-591233, 936246-591234, 936246-591242, 9..."
647,ocds-dgv273-seacev3-2023-1068-31,936246-591240,"936246-591233, 936246-591234, 936246-591242, 9..."
2828,ocds-dgv273-seacev3-1076786,1076786-20601573416,"1076786-10176025315, 1076786-10091877045, 1076..."
2830,ocds-dgv273-seacev3-1076786,1076786-10436316122,"1076786-10176025315, 1076786-10091877045, 1076..."
2831,ocds-dgv273-seacev3-1076786,1076786-10400908848,"1076786-10176025315, 1076786-10091877045, 1076..."


### NOTA DE CALIDAD - Contratos sin adjudicación en com_awards

Hallazgo
- 1.933 contratos (0,98% de 196.827) en 663 procesos (0,39% de 171.709 procesos con contratos) referencian un award_key que no existe en com_awards.
- Peso económico: 583,8 M de 197.355,5 M en contract_amount (0,30%). contracts no tiene moneda, por lo que la cifra es referencial si hay monedas mezcladas.
- Concentración: 4 procesos aportan el 38% de los casos y 20 procesos el 55%.

Evidencia de que no es un error de limpieza
- awards: 203.498 filas leídas y 203.498 conservadas (0 sin llave, 0 repetidas).
- Los ids están bien formados (p. ej. 937629-20609735601 frente a 937629-20530147551).
- Las adjudicaciones referidas no existen en los CSV de origen.

Causa
- Inconsistencia en el origen de la data.

Decisión
- Se conservan (ELIMINAR_HUERFANOS = False). La FK contracts.award_key -> awards.award_key está declarada y PRAGMA foreign_key_check lista los 1.933 casos.
- Motivos: son contratos reales; el impacto monetario es del 0,3%; eliminarlos sesgaría contra los procesos más grandes; el filtrado es reversible y el borrado no.

Impacto en los análisis
- Estos contratos no tienen proveedor vía awa_suppliers: los rankings por proveedor los subestiman.
- Cruces contracts -> awards: usar LEFT JOIN. Un INNER JOIN los descarta sin avisar.

Cómo identificarlos o excluirlos
- SELECT * FROM contracts c WHERE NOT EXISTS (SELECT 1 FROM awards a WHERE a.award_key = c.award_key);
- O filtrar con award_en_awards = 0 si se agregó la columna.

In [11]:
tablas = {
    "records": (records, "ocid"),
    "com_parties": (com_parties, "party_key"),
    "com_ten_tenderers": (com_ten_tenderers, "tenderer_key"),
    "com_awards": (com_awards, "award_key"),
    "com_contracts": (com_contracts, "contract_key"),
    "com_awa_suppliers": (com_awa_suppliers, "supplier_key"),
}

for nombre, (df, pk) in tablas.items():
    assert df[pk].is_unique, f"{nombre}: {pk} tiene repetidos"

# porcentaje de nulos por columna
for nombre, (df, _) in tablas.items():
    nulos = (df.isna().mean() * 100).round(1)
    print(f"\n{nombre} ({len(df):,} filas)")
    print(nulos[nulos > 0].to_string() if (nulos > 0).any() else "sin nulos")


records (236,234 filas)
budget_amount           27.3
procurement_method      14.6
tender_start_date        6.4
tender_end_date          6.4
number_of_tenderers      9.0
tender_duration_days     6.4

com_parties (3,143,082 filas)
region        92.5
department    92.5

com_ten_tenderers (2,906,843 filas)
sin nulos

com_awards (203,498 filas)
sin nulos

com_contracts (196,827 filas)
contract_title         0.2
final_value_amount    88.6
variacion_monto       88.6

com_awa_suppliers (199,581 filas)
sin nulos


## Carga a SQLite

Las fechas se guardan como texto `YYYY-MM-DD HH:MM:SS` para que funcionen `date()` y `strftime()` en SQL, y las banderas como 0/1.

In [13]:
def preparar_sql(df):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif pd.api.types.is_bool_dtype(out[c]):
            out[c] = out[c].astype("Int8")
    return out


PK = {
    "records": "ocid",
    "parties": "party_key",
    "ten_tenderers": "tenderer_key",
    "awards": "award_key",
    "contracts": "contract_key",
    "awa_suppliers": "supplier_key",
}
REL = {
    "parties": [("ocid", "records", "ocid")],
    "ten_tenderers": [("ocid", "records", "ocid")],
    "awards": [("ocid", "records", "ocid")],
    "contracts": [("award_key", "awards", "award_key")],
    "awa_suppliers": [("award_key", "awards", "award_key")],
}


def tipo_sql(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "INTEGER"
    if pd.api.types.is_float_dtype(serie):
        return "REAL"
    return "TEXT"   # texto y fechas (ya convertidas a texto)


def crear_tabla(conn, nombre, df):
    defs = []
    for c in df.columns:
        extra = " PRIMARY KEY NOT NULL" if c == PK[nombre] else ""
        defs.append(f'"{c}" {tipo_sql(df[c])}{extra}')
    for col, padre, col_padre in REL.get(nombre, []):
        defs.append(f'FOREIGN KEY ("{col}") REFERENCES "{padre}" ("{col_padre}")')
    conn.execute(f'CREATE TABLE "{nombre}" (\n  ' + ",\n  ".join(defs) + "\n)")


conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF")   # durante la carga; las FK se revisan al final

# se borran primero las tablas hijas
for t in ["awa_suppliers", "contracts", "ten_tenderers", "parties", "awards", "records"]:
    conn.execute(f'DROP TABLE IF EXISTS "{t}"')

# el orden de `tablas` ya pone a cada padre antes que sus hijas
for nombre, (df, _) in tablas.items():
    sql = nombre.replace("com_", "")
    datos = preparar_sql(df)
    crear_tabla(conn, sql, datos)
    datos.to_sql(sql, conn, if_exists="append", index=False)
    print(f"{sql}: {len(datos):,} filas cargadas")

# índices para los joins por FK (las PK ya traen su índice único)
for t, rel in REL.items():
    for col, _, _ in rel:
        conn.execute(f'CREATE INDEX IF NOT EXISTS ix_{t}_{col} ON "{t}"("{col}")')
conn.commit()

# filas que incumplen alguna FK
fallas = conn.execute("PRAGMA foreign_key_check").fetchall()
print(f"\nfilas que incumplen alguna FK: {len(fallas):,}")
conn.close()
print(f"Listo. Base de datos en: {DB_PATH}")

records: 236,234 filas cargadas
parties: 3,143,082 filas cargadas
ten_tenderers: 2,906,843 filas cargadas
awards: 203,498 filas cargadas
contracts: 196,827 filas cargadas
awa_suppliers: 199,581 filas cargadas

filas que incumplen alguna FK: 1,933
Listo. Base de datos en: ..\..\contrataciones.db


In [ ]:
# Creación de vistas sql para facilitar consultas en powerbi y otros clientes

conn = sqlite3.connect(DB_PATH)
with open(os.path.join("views.sql"), "r", encoding="utf-8") as f:
    conn.executescript(f.read())
conn.close()
print("Vistas creadas.")

Vistas creadas.
